<a href="https://colab.research.google.com/github/nandini2405/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nandini2405/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


### Rule

I will prioritize pages that have sufficient search visibility and relatively low CTR for their search position.

The rule will produce a score that ranks pages by review priority.

### Reason code

- `HIGH_VISIBILITY_LOW_CTR` — the page has substantial search impressions but a relatively low CTR for its search position.

In [3]:
import os
import duckdb
from google.colab import userdata
import pandas as pd
token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = token

con = duckdb.connect()

con.execute("SET VARIABLE hf_token = ?", [token])

con.execute("""
CREATE OR REPLACE SECRET hf_secret (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

print("DuckDB connection ready.")

DuckDB connection ready.


In [4]:
# Load March 2026 warehouse data
march_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

df = con.sql(f"""
SELECT *
FROM read_parquet('{march_path}')
""").df()

# Calculate CTR
df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"]

# Avoid invalid CTR values where impressions are zero
df.loc[df["gsc_impressions"] == 0, "ctr"] = None

print("Rows:", len(df))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 9841378


In [5]:
# Signal 1: Search visibility / impressions

df["impression_bucket"] = pd.cut(
    df["gsc_impressions"],
    bins=[-1, 99, 499, 999, float("inf")],
    labels=["Low", "Medium", "High", "Very High"]
)

impression_check = (
    df["impression_bucket"]
    .value_counts()
    .sort_index()
    .reset_index()
)

impression_check.columns = ["impression_bucket", "n"]

impression_check

,impression_bucket,n
0,Low,9202770
1,Medium,537157
2,High,69032
3,Very High,32419


In [11]:
impression_vs_ctr_mean = (
    df.groupby("impression_bucket", observed=True)
      .agg(
          n=("gsc_impressions", "size"),
          mean_ctr=("ctr", "mean")
      )
      .reset_index()
)

impression_vs_ctr_mean

,impression_bucket,n,mean_ctr
0,Low,9202770,0.003086
1,Medium,537157,0.003101
2,High,69032,0.002846
3,Very High,32419,0.002714


In [13]:
impression_ctr_activity = (
    df.groupby("impression_bucket", observed=True)
      .agg(
          n=("ctr", "size"),
          nonzero_ctr_rate=("ctr", lambda x: (x > 0).mean())
      )
      .reset_index()
)

impression_ctr_activity

,impression_bucket,n,nonzero_ctr_rate
0,Low,9202770,0.015930
1,Medium,537157,0.374859
2,High,69032,0.645903
3,Very High,32419,0.784447


In [7]:
df["position_bucket"] = pd.cut(
    df["gsc_avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["Top 3", "4-10", "11-20", "20+"]
)

position_check = (
    df["position_bucket"]
    .value_counts()
    .sort_index()
    .reset_index()
)

position_check.columns = ["position_bucket", "n"]

position_check

,position_bucket,n
0,Top 3,564173
1,4-10,1456122
2,11-20,519223
3,20+,908354


In [9]:
position_vs_ctr_mean = (
    df.groupby("position_bucket", observed=True)
      .agg(
          n=("gsc_avg_position", "size"),
          mean_ctr=("ctr", "mean")
      )
      .reset_index()
)

position_vs_ctr_mean

,position_bucket,n,mean_ctr
0,Top 3,564173,0.004918
1,4-10,1456122,0.003473
2,11-20,519223,0.002770
3,20+,908354,0.001289


In [17]:
position_ctr_activity = (
    df.groupby("position_bucket", observed=True)
      .agg(
          n=("ctr", "size"),
          nonzero_ctr_rate=("ctr", lambda x: (x > 0).mean())
      )
 .reset_index()
)
position_ctr_activity

,position_bucket,n,nonzero_ctr_rate
0,Top 3,564173,0.162158
1,4-10,1456122,0.152241
2,11-20,519223,0.103137
3,20+,908354,0.055168


### Signal verdicts

**1. Search impressions — CONFIRMED**

Higher impression buckets showed a much higher rate of non-zero CTR. This supports using impressions as a visibility/activity signal. However, higher impressions did not mean higher average CTR, so impressions should not be interpreted as a direct indicator of CTR quality.

**2. Average position — CONFIRMED**

CTR activity decreased as average search position became worse. This supports using search position as context when identifying pages with relatively low CTR.

These are observed relationships in the March 2026 daily warehouse data and do not establish causation.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [18]:
page_df = (
    df.groupby(["client_hash_id", "content_hash_id"])
      .agg(
          impressions=("gsc_impressions", "sum"),
          clicks=("gsc_clicks", "sum"),
          sum_position=("gsc_sum_position", "sum")
      )
      .reset_index()
)

page_df["ctr"] = (
    page_df["clicks"] / page_df["impressions"]
)

page_df["avg_position"] = (
    page_df["sum_position"] / page_df["impressions"]
)

print("Pages:", len(page_df))
page_df.head()

Pages: 331437


,client_hash_id,content_hash_id,impressions,clicks,sum_position,ctr,avg_position
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,0,NaN,NaN
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,0,NaN,NaN
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,0,NaN,NaN
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9,0.0,9.0
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,0,NaN,NaN


In [19]:
page_df["visibility_bucket"] = pd.cut(
    page_df["impressions"],
    bins=[-1, 0, 99, 499, 999, float("inf")],
    labels=["No impressions", "Low", "Medium", "High", "Very High"]
)

visibility_check = (
    page_df["visibility_bucket"]
    .value_counts()
    .sort_index()
    .reset_index()
)

visibility_check.columns = ["visibility_bucket", "n"]

visibility_check

,visibility_bucket,n
0,No impressions,154699
1,Low,75297
2,Medium,39517
3,High,16866
4,Very High,45058


In [20]:
page_df["position_bucket"] = pd.cut(
    page_df["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["Top 3", "4-10", "11-20", "20+"]
)

page_df["position_bucket"].value_counts().sort_index()

,count
position_bucket,
Top 3,17426
4-10,83288
11-20,29922
20+,44668


In [21]:
position_benchmark = (
    page_df.groupby("position_bucket", observed=True)
    .agg(
        n=("ctr", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean")
    )
    .reset_index()
)

position_benchmark

,position_bucket,n,median_ctr,mean_ctr
0,Top 3,17426,0.0,0.009961
1,4-10,83288,0.0,0.004873
2,11-20,29922,0.0,0.003285
3,20+,44668,0.0,0.001952


In [27]:
page_df["position_benchmark_ctr"] = pd.to_numeric(
    page_df["position_benchmark_ctr"],
    errors="coerce"
)

page_df["ctr_gap"] = (
    page_df["position_benchmark_ctr"] - page_df["ctr"]
)

page_df[[
    "impressions",
    "ctr",
    "position_benchmark_ctr",
    "ctr_gap"
]].head(10)

,impressions,ctr,position_benchmark_ctr,ctr_gap
0,0,NaN,NaN,NaN
1,0,NaN,NaN,NaN
2,0,NaN,NaN,NaN
3,1,0.000000,0.004873,0.004873
4,0,NaN,NaN,NaN
5,0,NaN,NaN,NaN
6,0,NaN,NaN,NaN
7,331,0.006042,0.003285,-0.002758
8,33,0.000000,0.004873,0.004873
9,0,NaN,NaN,NaN


In [29]:
ranked_df = page_df[
    (page_df["impressions"] >= 500) &
    (page_df["ctr_gap"] > 0)
].copy()

print("Pages selected:", len(ranked_df))

ranked_df[[
    "client_hash_id",
    "content_hash_id",
    "impressions",
    "ctr",
    "avg_position",
    "position_benchmark_ctr",
    "ctr_gap"
]].head(10)

Pages selected: 49564


,client_hash_id,content_hash_id,impressions,ctr,avg_position,position_benchmark_ctr,ctr_gap
178,client_0797ff3a1fc9a6a5,content_be06033d30b49299,2092,0.000478,53.148184,0.001952,0.001474
281,client_08a6a72ff48e62c0,content_003ecfeb2a98c909,2270,0.000000,9.352863,0.004873,0.004873
302,client_08a6a72ff48e62c0,content_005de32a6050545f,1604,0.001247,35.267456,0.001952,0.000706
315,client_08a6a72ff48e62c0,content_0075b9591f39fcad,4991,0.004408,6.358646,0.004873,0.000465
322,client_08a6a72ff48e62c0,content_008538a5278580a8,4991,0.004408,6.358646,0.004873,0.000465
327,client_08a6a72ff48e62c0,content_0093a097f50ea763,18246,0.002686,5.455881,0.004873,0.002187
345,client_08a6a72ff48e62c0,content_00c0f84b65018aea,4991,0.004408,6.358646,0.004873,0.000465
380,client_08a6a72ff48e62c0,content_010755391b1f8d46,5972,0.001674,5.847790,0.004873,0.003198
393,client_08a6a72ff48e62c0,content_011ad9c4e13ce8ee,1535,0.001303,3.203909,0.004873,0.003570
397,client_08a6a72ff48e62c0,content_01211c8b0b0ebf7c,4991,0.004408,6.358646,0.004873,0.000465


In [30]:
import numpy as np

ranked_df["score"] = (
    np.log1p(ranked_df["impressions"]) *
    ranked_df["ctr_gap"]
)

ranked_df[[
    "client_hash_id",
    "content_hash_id",
    "impressions",
    "ctr",
    "position_benchmark_ctr",
    "ctr_gap",
    "score"
]].head(10)

,client_hash_id,content_hash_id,impressions,ctr,position_benchmark_ctr,ctr_gap,score
178,client_0797ff3a1fc9a6a5,content_be06033d30b49299,2092,0.000478,0.001952,0.001474,0.011274
281,client_08a6a72ff48e62c0,content_003ecfeb2a98c909,2270,0.000000,0.004873,0.004873,0.037657
302,client_08a6a72ff48e62c0,content_005de32a6050545f,1604,0.001247,0.001952,0.000706,0.005208
315,client_08a6a72ff48e62c0,content_0075b9591f39fcad,4991,0.004408,0.004873,0.000465,0.003959
322,client_08a6a72ff48e62c0,content_008538a5278580a8,4991,0.004408,0.004873,0.000465,0.003959
327,client_08a6a72ff48e62c0,content_0093a097f50ea763,18246,0.002686,0.004873,0.002187,0.021461
345,client_08a6a72ff48e62c0,content_00c0f84b65018aea,4991,0.004408,0.004873,0.000465,0.003959
380,client_08a6a72ff48e62c0,content_010755391b1f8d46,5972,0.001674,0.004873,0.003198,0.027810
393,client_08a6a72ff48e62c0,content_011ad9c4e13ce8ee,1535,0.001303,0.004873,0.003570,0.026192
397,client_08a6a72ff48e62c0,content_01211c8b0b0ebf7c,4991,0.004408,0.004873,0.000465,0.003959


In [31]:
ranked_df = ranked_df.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

ranked_df["rank"] = ranked_df.index + 1

ranked_df[[
    "rank",
    "client_hash_id",
    "content_hash_id",
    "impressions",
    "ctr",
    "avg_position",
    "ctr_gap",
    "score"
]].head(20)

,rank,client_hash_id,content_hash_id,impressions,ctr,avg_position,ctr_gap,score
0,1,client_23a62021009f63c4,content_44f34c0a90047651,212404,0.000113,0.665877,0.009848,0.120800
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984,0.000007,2.693038,0.009954,0.117583
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075,0.000008,0.308426,0.009953,0.116736
3,4,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83834,0.000012,0.116003,0.009949,0.112790
4,5,client_e547b89c05043229,content_306bc78dff1eb683,80821,0.000433,1.444266,0.009528,0.107667
5,6,client_23a62021009f63c4,content_bf078007df823490,44707,0.000000,1.400049,0.009961,0.106663
6,7,client_62f4a7e64f5e0096,content_fc67675904376267,60172,0.000299,2.126022,0.009662,0.106330
7,8,client_1a730cb2640a1abf,content_d61fc394d10cba41,38000,0.000026,2.362579,0.009935,0.104766
8,9,client_e547b89c05043229,content_c46df0fa61530d86,70398,0.000597,0.969332,0.009365,0.104526
9,10,client_e547b89c05043229,content_8d7d99f109e19aa2,203497,0.001420,2.468557,0.008541,0.104400


In [33]:
ranked_df["reason_code"] = "HIGH_VISIBILITY_LOW_CTR"

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.